# 共同随机趋势与平稳价差
先修：AR(1)、OLS、单位根和协整的定义。先区分确定性算例与随机模型，再估计长期关系、误差修正和 OU 时间尺度，最后重复模拟看不确定性。

原来的 $0.55^t$ 是没有创新的确定性衰减，只适合核对代数，不是平稳 AR(1) 样本。下面先保留这种形状，再生成有独立正态创新、正确平稳初值的价差。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen
rng = np.random.default_rng(2026)
n, phi, noise = 300, .8, .5
a, beta = 1.5, 2.
deterministic = .55 ** np.arange(30)
x = np.cumsum(rng.normal(0, 1, n))
z = np.empty(n)
z[0] = rng.normal(0, noise/np.sqrt(1-phi**2))
for t in range(1, n):
    z[t] = phi*z[t-1] + rng.normal(0, noise)
y = a + beta*x + z
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(deterministic); axes[0].set_title('deterministic decay')
axes[1].plot(z); axes[1].set_title('stationary AR(1) realization'); plt.show()

初值方差 $\sigma_\eta^2/(1-\phi^2)$ 使整个残差过程平稳。随机游走 $x$ 的方差随时间增长，而 $y-1.5-2x=z$ 的方差恒定。先预测：OLS 是否每次都恰好恢复斜率 2？

In [ ]:
design = np.column_stack([np.ones(n), x])
estimated = np.linalg.lstsq(design, y, rcond=None)[0]
residual = y-design@estimated
print('截距、斜率：', estimated)
# 事先选定含常数、最多 1 个差分滞后，关闭自动选阶。
eg_stat, eg_p, eg_critical = coint(y, x, trend='c', maxlag=1, autolag=None)
print('Engle-Granger 统计量、p 值：', eg_stat, eg_p)
print('真实 z 的 ADF（另一检验问题）：', adfuller(z, regression='c', maxlag=1, autolag=None)[:2])
fig, ax = plt.subplots(); ax.plot(z, label='true spread'); ax.plot(residual, label='estimated spread', alpha=.7)
ax.legend(); plt.show()

估计残差的单位根检验要使用协整检验的临界值，不能将普通 ADF 的 p 值直接当作 Engle–Granger 的 p 值。两格结果都有有限样本不确定性。多变量情形的 Johansen 检验将长期关系数目变成秩问题；下格仅作选读比较。

In [ ]:
johansen = coint_johansen(np.column_stack([y, x]), det_order=0, k_ar_diff=1)
print('Johansen trace：', johansen.lr1, '95% 临界值：', johansen.cvt[:, 1])
ecm_design = np.column_stack([np.ones(n-1), residual[:-1], np.diff(x)])
ecm = np.linalg.lstsq(ecm_design, np.diff(y), rcond=None)[0]
ar = np.linalg.lstsq(np.column_stack([np.ones(n-1), residual[:-1]]), residual[1:], rcond=None)[0]
estimated_phi = ar[1]
print('ECM 调整系数（理论 phi-1）：', ecm[1], phi-1)
if 0 < estimated_phi < 1:
    kappa = -np.log(estimated_phi)  # 采样间隔 1
    innovations = residual[1:] - ar[0] - estimated_phi*residual[:-1]
    sigma_ou = np.sqrt(innovations.var(ddof=2)*2*kappa/(1-estimated_phi**2))
    print('OU kappa、扩散率、半衰期：', kappa, sigma_ou, np.log(2)/kappa)
else:
    print('此样本估计不满足正均值回归 OU 的映射条件。')

## 拒绝频率：功效与第一类错误
存在协整的模型比较样本量和持续性；两条独立随机游走没有协整，拒绝频率估计第一类错误。无协整时没有真实长期斜率可供估计，不将其回归斜率称为参数误差。

每个情形内独立重复 $B$ 次，报告拒绝频率 $\hat p$、模拟标准误 $\sqrt{\hat p(1-\hat p)/B}$ 和 Wilson 95% 区间。后者在全部拒绝或全部不拒绝时仍给出非零宽度。区间描述有限次模拟的不确定性，不是“存在协整的概率”。

Wilson 区间通过解不等式 $|\hat p-p|\le 1.96\sqrt{p(1-p)/B}$ 得到，把未知 $p$ 保留在方差项中再求区间，避免把观察到的 0 或 1 当作没有不确定性。

In [ ]:
def one_sample(count, persistence, generator, cointegrated=True):
    xx = np.cumsum(generator.normal(size=count))
    if cointegrated:
        zz = np.empty(count)
        zz[0] = generator.normal(0, .5/np.sqrt(1-persistence**2))
        for t in range(1,count):
            zz[t] = persistence*zz[t-1]+generator.normal(0,.5)
        yy = 1.5+2*xx+zz
        slope_error = np.linalg.lstsq(np.column_stack([np.ones(count),xx]),yy,rcond=None)[0][1]-2
    else:
        yy = np.cumsum(generator.normal(size=count))
        slope_error = np.nan
    statistic,pvalue,_ = coint(yy,xx,trend='c',maxlag=1,autolag=None)
    return slope_error,pvalue

repeats = 300  # 可修改；增加重复次数减小 Monte Carlo 误差。
alpha = .05
scenarios = [('power n80',80,.8,True),('power n300',300,.8,True),
             ('power phi.97',300,.97,True),('size n80',80,.8,False),
             ('size n300',300,.8,False)]
streams = np.random.SeedSequence(20260908).spawn(len(scenarios))
summary = []
for (label,count,persistence,cointegrated),seed in zip(scenarios,streams):
    generator = np.random.default_rng(seed)
    draws = np.array([one_sample(count,persistence,generator,cointegrated) for _ in range(repeats)])
    frequency = np.mean(draws[:,1]<alpha)
    mcse = np.sqrt(frequency*(1-frequency)/repeats)
    z95 = 1.959963984540054
    denominator = 1+z95**2/repeats
    center = (frequency+z95**2/(2*repeats))/denominator
    half = z95*np.sqrt(frequency*(1-frequency)/repeats+z95**2/(4*repeats**2))/denominator
    low,high = center-half,center+half
    summary.append((label,frequency,mcse,low,high))
    print(label,'frequency / MCSE / Wilson:',frequency,mcse,(low,high))
    if cointegrated:
        print('斜率误差 10/50/90 分位:',np.quantile(draws[:,0],[.1,.5,.9]))
fig,ax=plt.subplots(figsize=(9,3))
frequencies=np.array([row[1] for row in summary])
positions=np.arange(len(summary))
ax.vlines(positions,[row[3] for row in summary],[row[4] for row in summary])
ax.scatter(positions,frequencies)
ax.axhline(alpha,color='tab:red',linestyle='--',label='nominal size .05')
ax.set(xticks=np.arange(len(summary)),xticklabels=[row[0] for row in summary],ylabel='rejection frequency',ylim=(0,1.02))
ax.legend();plt.show()

## 噪声尺度究竟改变什么
固定同一条随机游走和标准化平稳价差 $z$，只改变 $y=D\beta+cz$ 中的 $c>0$。投影给出
$$\hat\beta_c-\beta=cD^+z,\qquad\hat u_c=c(I-P_D)z.$$
因此系数误差按 $c$ 缩放。固定差分滞后、确定性项和样本长度时，DF 回归的响应和解释变量一起缩放，系数及其 t 统计量不变；正式协整检验也应近似不变。这里不加入独立测量噪声或重新选择样本，避免同时改变其他对象。

In [ ]:
paired_rng=np.random.default_rng(724)
paired_count,paired_phi=300,.8
paired_x=np.cumsum(paired_rng.normal(size=paired_count))
standard_z=np.empty(paired_count)
standard_z[0]=paired_rng.normal()/np.sqrt(1-paired_phi**2)
standard_innovations=paired_rng.normal(size=paired_count-1)
for t in range(1,paired_count):
    standard_z[t]=paired_phi*standard_z[t-1]+standard_innovations[t-1]
paired_design=np.column_stack([np.ones(paired_count),paired_x])
base_parameters=np.array([1.5,2.])
paired_results=[]
for scale in [.5,2.]:
    paired_y=paired_design@base_parameters+scale*standard_z
    coefficients=np.linalg.lstsq(paired_design,paired_y,rcond=None)[0]
    paired_residual=paired_y-paired_design@coefficients
    statistic,pvalue,_=coint(paired_y,paired_x,trend='c',maxlag=1,autolag=None)
    paired_results.append((coefficients,paired_residual,statistic,pvalue))
    print('scale / slope error / EG statistic / p:',scale,coefficients[1]-2,statistic,pvalue)
print('参数误差的四倍缩放差:',paired_results[1][0]-base_parameters-4*(paired_results[0][0]-base_parameters))
print('残差的四倍缩放最大差:',np.max(abs(paired_results[1][1]-4*paired_results[0][1])))
print('统计量之差:',paired_results[1][2]-paired_results[0][2])

固定规格的尺度不变性是这个生成模型的性质，不意味着噪声在任何模型下都不影响检验力。加入独立测量误差、改变信噪成分的相对结构或使用不同滞后规格后，前述投影等式不再直接解释全部变化。数值上极端尺度也可能损害精度。

### 半衰期与一个透明 DF 回归
下格用不含常数、零差分滞后的同一规格比较回归统计量；估计残差的正式协整显著性仍用前面的专用检验。

In [ ]:
phis = np.linspace(.01, .99, 100)
fig, ax = plt.subplots()
ax.plot(phis, -np.log(2)/np.log(phis))
ax.set(xlabel='phi (valid OU range: 0 < phi < 1)', ylabel='half-life')
plt.show()
lag = residual[:-1]
dy = np.diff(residual)
gamma = (lag @ dy) / (lag @ lag)
error = dy-gamma*lag
se = np.sqrt((error @ error)/(len(dy)-1)/(lag @ lag))
print('透明 DF / 相同规格 adfuller：', gamma/se,
      adfuller(residual, maxlag=0, regression='n', autolag=None)[0])

## 练习与反馈
1. 哪些行估计功效，哪些行估计第一类错误？为什么无协整行没有“真实斜率误差”？
2. 若拒绝率约为 0.5，60 次与 300 次重复的模拟标准误各是多少？
3. 相同创新下把 $c$ 从 0.5 改为 2，哪些对象变为四倍，哪些应不变？

**反馈。** 共同趋势加平稳价差的行估计功效；独立随机游走的行估计第一类错误，其总体不存在协整斜率。标准误分别约为 0.0645 与 0.0289，几百分点差异不足以自动认定功效不同；边界频率还需看区间而非零插件标准误。参数估计误差与残差变为四倍，固定规格的统计量保持不变。

$\phi$ 接近 1 时冲击衰减慢，短路径类似随机游走。半衰期是条件均值偏离减半的时间，不是随机首次命中均值的平均时间。改变 $\phi$ 且固定创新方差还会改变平稳方差，应把持续性效应与纯粹整体缩放区别开。

接口说明：[NumPy 最小二乘](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)、[statsmodels 协整检验](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html)。